define targets

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
CRYPTO_TARGET = 'TINYAES128C'
VERSION = 'HARDWARE'

In [2]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen            

compile and download firmware

In [3]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-aes
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

Building for platform CWHUSKY with CRYPTO_TARGET=TINYAES128C
SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
Blank crypto options, building for AES128
arm-none-eabi-gcc (15:14.2.rel1-1) 14.2.1 20241119
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
Compiling:
Compiling:
Compiling:
.
-en     simpleserial-aes.c ...
-en     .././simpleserial/simpleserial.c ...
-en     .././hal/hal.c ...
Compiling:
.
-en     .././hal//sam4s/startup_sam4s.c ...
Compiling:
.
.
-en     .././hal//sam4s/sam4s_hal.c ...
Compiling:
.
Compiling:
-en     .././hal//sam4s/uart.c ...
.
-en     .././hal//sam4s/pio.c ...
Compiling:
Compiling:
.
-en     .././hal//sam4s/system_sam4s.c ...
-en     .././hal//sam4s/sysclk.c ...
Compiling:
.
-en     .././hal//sam4s/pmc.c ...
Compilin

In [4]:
cw.program_target(scope, prog, "../../../firmware/mcu/simpleserial-aes/simpleserial-aes-{}.hex".format(PLATFORM))

capture traces

In [41]:
from tqdm.notebook import trange
import numpy as np
import time
import numpy as np

def CaptureAndPlot():
    ktp = cw.ktp.Basic()
    trace_array = []
    textin_array = []
    
    key, text = ktp.next()

    print(len(text))
    print(text)
    
    target.set_key(key)
    
    N = 100
    for i in trange(N, desc='Capturing traces'):
        scope.arm()
        if text[00] & 0x01:
            text[15] = 0xFF
        else:
            text[15] = 0x00
        target.simpleserial_write('p', text)
        
        ret = scope.capture()
        if ret:
            print("Target timed out!")
            continue
        
        response = target.simpleserial_read('r', 16)
        
        trace_array.append(scope.get_last_trace())
        textin_array.append(text)
        
        key, text = ktp.next() 
    
    one_list = []
    zero_list = []
    for i in range(len(trace_array)):
        if textin_array[i][0] == 0x00:
            zero_list.append(trace_array[i])
        else:
            one_list.append(trace_array[i])
    
    one_avg = np.mean(one_list, axis=0)
    zero_avg = np.mean(zero_list, axis=0)
    
    diff = one_avg - zero_avg
    return diff

diff = CaptureAndPlot()
cw.plot(diff)

16
CWbytearray(b'a9 98 b4 3d 9e d2 5e 2f 6e a1 df 1b 4a 61 56 65')


Capturing traces:   0%|          | 0/100 [00:00<?, ?it/s]

:Curve   [x]   (y)